# MambaVision-T — Indiana University Chest X-ray | BASELINE (Kaggle)
### Stage 1-4 UNMODIFIED | Full benchmark — base paper results

**Dataset:** Indiana University Chest X-ray (Kaggle)
`chest-xrays-indiana-university` — 7,470 frontal X-rays, 14 disease classes

**MambaVision:** Upload your `MambaVision/` folder to `/kaggle/input/` as a dataset.
No pip install needed — folder is added to sys.path directly.

**Sequence (identical to experiment notebook):**

| Cell | Content | Note |
|---|---|---|
| 1 | Imports | installs fvcore if missing |
| 2 | PyTorch 2.6 patch | |
| 3 | MambaVision path (from uploaded dataset) | no pip install |
| 4 | Dataset class + record builder | Indiana-specific |
| 5 | DataLoaders | 80/20 split |
| 6 | Load model — Stage 1-4 original | |
| 7 | FLOPs & params | before training |
| 8 | Loss / Optimiser / Scheduler | dynamic pos_weight |
| 9 | Train + Validate functions | all 13 metrics |
| 10 | Training loop | GPU peak tracked here |
| 11 | Load best model | |
| 12 | Inference time | after training |
| 13 | GPU memory | after training |
| 14 | Final evaluation | all metrics |
| 15 | Training curves | 3×3 grid |
| 16 | Per-class visualisation | bars + radar + heatmap |
| 17 | Benchmark summary | |
| 18 | Save JSON + TXT | /kaggle/working/ |

## Cell 1 — Imports & Environment

In [1]:
import os, sys, time, json, warnings, math, random, shutil
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import cv2
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms

from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score,
    matthews_corrcoef, hamming_loss
)
from sklearn.model_selection import train_test_split

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

# ── Install & import FLOPs library ───────────────────────────────────
FLOPS_AVAILABLE = False
THOP_AVAILABLE  = False
try:
    from fvcore.nn import FlopCounterMode
    FLOPS_AVAILABLE = True
    print("✓ fvcore — exact FLOPs")
except ImportError:
    try:
        os.system("pip install fvcore -q")
        from fvcore.nn import FlopCounterMode
        FLOPS_AVAILABLE = True
        print("✓ fvcore installed & imported")
    except Exception:
        try:
            from thop import profile as thop_profile
            THOP_AVAILABLE = True
            print("✓ thop — approx FLOPs")
        except ImportError:
            print("⚠ no FLOPs library — will use paper estimate")

# ── Environment ───────────────────────────────────────────────────────
print(f"\nPyTorch  : {torch.__version__}")
print(f"CUDA ver : {torch.version.cuda}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    TOTAL_VRAM_MB = props.total_memory / 1024**2
    print(f"GPU      : {props.name}")
    print(f"VRAM     : {TOTAL_VRAM_MB/1024:.2f} GB  ({TOTAL_VRAM_MB:.0f} MB)")
else:
    TOTAL_VRAM_MB = 0

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device   : {device}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.9 MB/s eta 0:00:00
⚠ no FLOPs library — will use paper estimate

PyTorch  : 2.10.0+cu128
CUDA ver : 12.8
CUDA OK  : True
GPU      : Tesla T4
VRAM     : 14.56 GB  (14913 MB)
Device   : cuda


## Cell 2 — PyTorch 2.6 Patch

In [2]:
import argparse
try:
    import torch.serialization
    torch.serialization.add_safe_globals([argparse.Namespace])
    print("✓ argparse.Namespace allowlisted (PyTorch 2.6 fix)")
except AttributeError:
    print("✓ PyTorch < 2.6 — no patch needed")

_orig_load = torch.load
def _safe_load(f, map_location=None, pickle_module=None, weights_only=False, **kw):
    return _orig_load(f, map_location=map_location, weights_only=False, **kw)
torch.load = _safe_load
print("✓ torch.load monkey-patched → weights_only=False")

✓ argparse.Namespace allowlisted (PyTorch 2.6 fix)
✓ torch.load monkey-patched → weights_only=False


## Cell 3 — MambaVision Path Setup
> Upload your `MambaVision` folder to Kaggle as a dataset (e.g. `your-username/mambavision-repo`).
> It will appear at `/kaggle/input/mambavision-repo/MambaVision/`.
> **No pip install** — we add it directly to sys.path.

In [6]:
# ── Path to your uploaded MambaVision dataset on Kaggle ──────────────
# Upload MambaVision folder as Kaggle dataset, then set this path:
MAMBA_VISION_ROOT = '/kaggle/input/datasets/qaiserfarooq285/mambass/MambaVision'

# If uploaded differently, check with:
# import os; print(os.listdir('/kaggle/input/'))

for p in [MAMBA_VISION_ROOT, os.path.dirname(MAMBA_VISION_ROOT)]:
    if p not in sys.path:
        sys.path.insert(0, p)
    print(f"  sys.path ← {p}")

model_file = os.path.join(MAMBA_VISION_ROOT,'mambavision','models','mamba_vision.py')
print(f"\n  Model file exists : {os.path.exists(model_file)}")
print(f"  → {model_file}")

# Install mamba-ssm (required kernel dependency)
print("\nInstalling mamba-ssm...")
os.system("pip install mamba-ssm --no-build-isolation -q 2>&1 | tail -3")

try:
    from mamba_ssm import Mamba
    print("✓ mamba-ssm imported")
except ImportError as e:
    print(f"✗ mamba-ssm: {e}")
    print("  Try: !pip install mamba-ssm --no-build-isolation")

try:
    from mambavision.models.mamba_vision import mamba_vision_T
    import mambavision.models.mamba_vision as _mv
    print(f"✓ mamba_vision_T imported from: {_mv.__file__}")
    # Verify Stage 4 is original unmodified MambaVisionLayer
    _tmp = mamba_vision_T(pretrained=False, num_classes=14)
    s4   = type(_tmp.levels[3]).__name__
    assert s4 == 'MambaVisionLayer', f"Expected MambaVisionLayer but got {s4}"
    print(f"  ✓ Stage 4: {s4}  (UNMODIFIED baseline)")
    del _tmp
except Exception as e:
    print(f"✗ {e}")

  sys.path ← /kaggle/input/datasets/qaiserfarooq285/mambass/MambaVision
  sys.path ← /kaggle/input/datasets/qaiserfarooq285/mambass

  Model file exists : True
  → /kaggle/input/datasets/qaiserfarooq285/mambass/MambaVision/mambavision/models/mamba_vision.py

Installing mamba-ssm...
✓ mamba-ssm imported
✓ mamba_vision_T imported from: /kaggle/input/datasets/qaiserfarooq285/mambass/MambaVision/mambavision/models/mamba_vision.py
  ✓ Stage 4: MambaVisionLayer  (UNMODIFIED baseline)


## Cell 4 — Indiana Dataset Class

In [7]:
class IndianaChestXrayDataset(Dataset):
    """
    Indiana University Chest X-ray Dataset (Kaggle).
    Multi-label, 14 disease classes — same label set as NIH ChestX-ray14
    so all metric code stays identical.

    Kaggle dataset: chest-xrays-indiana-university
    Structure:
        /kaggle/input/chest-xrays-indiana-university/
            indiana_reports.csv    ← findings / impression text (not used for labels)
            indiana_projections.csv ← maps uid to image filename + projection
            images/images_normalized/  ← all PNG images

    Labels are parsed from the 'Problems' column in indiana_projections.csv
    which contains pipe-separated disease tags matching NIH class names.
    """

    ALL_CLASSES = [
        'Atelectasis','Consolidation','Infiltration','Pneumothorax',
        'Edema','Emphysema','Fibrosis','Effusion','Pneumonia',
        'Pleural_Thickening','Cardiomegaly','Nodule','Mass','Hernia'
    ]

    # Indiana dataset uses slightly different tag spellings — map them
    TAG_MAP = {
        'pleural effusion':      'Effusion',
        'effusion':              'Effusion',
        'pneumothorax':          'Pneumothorax',
        'cardiomegaly':          'Cardiomegaly',
        'edema':                 'Edema',
        'pulmonary edema':       'Edema',
        'consolidation':         'Consolidation',
        'infiltrate':            'Infiltration',
        'infiltration':          'Infiltration',
        'pneumonia':             'Pneumonia',
        'atelectasis':           'Atelectasis',
        'emphysema':             'Emphysema',
        'fibrosis':              'Fibrosis',
        'pleural thickening':    'Pleural_Thickening',
        'nodule':                'Nodule',
        'mass':                  'Mass',
        'hernia':                'Hernia',
        'hiatal hernia':         'Hernia',
    }

    def __init__(self, image_dir, records, transform=None, target_size=224):
        """
        records: list of dicts with keys 'filename' and 'labels' (np.array len 14)
        """
        self.image_dir   = image_dir
        self.records     = records
        self.transform   = transform
        self.target_size = target_size
        print(f"  Dataset: {len(self.records):,} images")

    def __len__(self): return len(self.records)

    def _resize_pad(self, img, sz):
        h, w  = img.shape[:2]
        scale = min(sz/h, sz/w)
        nh, nw = int(h*scale), int(w*scale)
        img = cv2.resize(img, (nw, nh), interpolation=cv2.INTER_LINEAR)
        canvas = np.zeros((sz, sz, 3), dtype=img.dtype)
        y0 = (sz-nh)//2; x0 = (sz-nw)//2
        canvas[y0:y0+nh, x0:x0+nw] = img
        return canvas

    def __getitem__(self, idx):
        rec  = self.records[idx]
        path = os.path.join(self.image_dir, rec['filename'])
        img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            # Try with .png extension if not found
            path = path if path.endswith('.png') else path + '.png'
            img  = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            img = np.zeros((224, 224), dtype=np.uint8)
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
        img = self._resize_pad(img, self.target_size)
        img = self.transform(img) if self.transform else               torch.from_numpy(img).permute(2,0,1).float()/255.
        return img, torch.from_numpy(rec['labels'])


def build_indiana_records(proj_csv_path, image_dir):
    """
    Parse indiana_projections.csv → list of records.
    Each record: {'filename': str, 'labels': np.float32 array len 14}
    Only keeps frontal (PA or AP) projections.
    """
    proj = pd.read_csv(proj_csv_path)
    print(f"  Projections CSV: {len(proj):,} rows")
    print(f"  Columns: {proj.columns.tolist()}")

    # Find the image filename and projection columns
    # Indiana CSV columns: uid, filename, projection, Problems (tags)
    fn_col  = [c for c in proj.columns if 'filename' in c.lower() or 'file' in c.lower()]
    tag_col = [c for c in proj.columns if 'problem' in c.lower() or 'label' in c.lower()
                                       or 'finding' in c.lower() or 'tag' in c.lower()]
    prj_col = [c for c in proj.columns if 'projection' in c.lower()]

    fn_col  = fn_col[0]  if fn_col  else proj.columns[1]
    tag_col = tag_col[0] if tag_col else proj.columns[-1]
    prj_col = prj_col[0] if prj_col else None

    print(f"  Using → filename='{fn_col}'  tags='{tag_col}'  projection='{prj_col}'")

    records = []
    tag_map = IndianaChestXrayDataset.TAG_MAP
    all_cls = IndianaChestXrayDataset.ALL_CLASSES

    for _, row in proj.iterrows():
        # Keep only frontal projections if column exists
        if prj_col and pd.notna(row[prj_col]):
            proj_val = str(row[prj_col]).lower()
            if not any(x in proj_val for x in ['frontal','pa','ap']):
                continue

        fname = str(row[fn_col]).strip()
        if not fname or fname == 'nan':
            continue

        # Build label vector
        labels = np.zeros(len(all_cls), dtype=np.float32)
        raw_tags = str(row[tag_col]) if pd.notna(row[tag_col]) else ''
        for tag in raw_tags.lower().replace(';','|').split('|'):
            tag = tag.strip()
            mapped = tag_map.get(tag)
            if mapped and mapped in all_cls:
                labels[all_cls.index(mapped)] = 1.0

        # Verify image exists
        img_path = os.path.join(image_dir, fname)
        if not os.path.exists(img_path):
            img_path2 = img_path + '.png'
            if not os.path.exists(img_path2):
                continue  # skip missing images

        records.append({'filename': fname, 'labels': labels})

    print(f"  Valid frontal records: {len(records):,}")
    pos_counts = np.array([r['labels'] for r in records]).sum(0)
    print(f"  Label distribution (pos counts per class):")
    for i, cn in enumerate(all_cls):
        print(f"    {cn:<22}: {int(pos_counts[i]):>4}")
    return records

print("✓ IndianaChestXrayDataset + build_indiana_records defined")

✓ IndianaChestXrayDataset + build_indiana_records defined


## Cell 5 — DataLoaders (Indiana, Kaggle paths)

In [14]:
import os
import random
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import cv2

# ── Paths ────────────────────────────────────────────────────────────
DATASET_DIR  = '/kaggle/input/datasets/raddar/chest-xrays-indiana-university'

IMAGE_DIR    = os.path.join(DATASET_DIR, 'images', 'images_normalized')
PROJ_CSV     = os.path.join(DATASET_DIR, 'indiana_projections.csv')
REPORT_CSV   = os.path.join(DATASET_DIR, 'indiana_reports.csv')

# ── Check paths ──────────────────────────────────────────────────────
print("Path check:")
for lbl, p in [('Images', IMAGE_DIR), ('Proj CSV', PROJ_CSV), ('Report CSV', REPORT_CSV)]:
    print(f"{'✓' if os.path.exists(p) else '✗'} {lbl}: {p}")

# ── Disease labels (14 classes) ─────────────────────────────────────
DISEASES = [
    "atelectasis","consolidation","infiltration","pneumothorax","edema",
    "emphysema","fibrosis","effusion","pneumonia","pleural_thickening",
    "cardiomegaly","nodule","mass","hernia"
]

# Improved keyword mapping (fix zero classes)
KEYWORDS = {
    "atelectasis": ["atelectasis"],
    "consolidation": ["consolidation"],
    "infiltration": ["infiltrate", "infiltration"],
    "pneumothorax": ["pneumothorax"],
    "edema": ["edema"],
    "emphysema": ["emphysema"],
    "fibrosis": ["fibrosis"],
    "effusion": ["effusion", "pleural effusion"],
    "pneumonia": ["pneumonia"],
    "pleural_thickening": ["pleural thickening"],
    "cardiomegaly": ["cardiomegaly", "enlarged heart"],
    "nodule": ["nodule"],
    "mass": ["mass"],
    "hernia": ["hernia"]
}

def extract_labels(text):
    text = str(text).lower()
    labels = np.zeros(len(DISEASES))
    for i, d in enumerate(DISEASES):
        for kw in KEYWORDS[d]:
            if kw in text:
                labels[i] = 1
                break
    return labels

# ── Build dataset records ───────────────────────────────────────────
def build_indiana_records(proj_csv, report_csv, image_dir):
    proj = pd.read_csv(proj_csv)
    rep  = pd.read_csv(report_csv)

    # Keep frontal only
    proj = proj[proj['projection'].str.lower() == 'frontal']

    # Merge
    df = pd.merge(proj, rep, on='uid')

    records = []

    for _, row in df.iterrows():
        img_path = os.path.join(image_dir, row['filename'])

        if not os.path.exists(img_path):
            continue

        text = str(row['findings']) + " " + str(row['impression'])
        labels = extract_labels(text)

        records.append({
            'image_path': img_path,
            'labels': labels,
            'text': text
        })

    print(f"\nValid records: {len(records)}")

    # Label stats
    label_sum = np.sum([r['labels'] for r in records], axis=0)
    print("\nLabel distribution:")
    for d, c in zip(DISEASES, label_sum):
        print(f"{d:20s}: {int(c)}")

    return records

# ── Dataset class ───────────────────────────────────────────────────
class IndianaDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        r = self.records[idx]

        img = cv2.imread(r['image_path'])

        # Handle corrupted images safely
        if img is None:
            img = np.zeros((224, 224, 3), dtype=np.uint8)

        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        if self.transform:
            img = self.transform(img)

        labels = torch.tensor(r['labels'], dtype=torch.float32)

        return img, labels

# ── Build records ───────────────────────────────────────────────────
all_records = build_indiana_records(PROJ_CSV, REPORT_CSV, IMAGE_DIR)

# ── Train/Val split ─────────────────────────────────────────────────
random.seed(42)
np.random.seed(42)

has_finding = [int(r['labels'].sum() > 0) for r in all_records]

train_recs, val_recs = train_test_split(
    all_records,
    test_size=0.2,
    random_state=42,
    stratify=has_finding if len(set(has_finding)) > 1 else None
)

print(f"\nTrain: {len(train_recs)} | Val: {len(val_recs)}")

# ── Transforms (🔥 FIXED resizing) ───────────────────────────────────
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

val_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

# ── Datasets ────────────────────────────────────────────────────────
train_ds = IndianaDataset(train_recs, train_tf)
val_ds   = IndianaDataset(val_recs, val_tf)

# ── DataLoaders (stable settings) ───────────────────────────────────
train_loader = DataLoader(
    train_ds,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_ds,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"\nTrain batches: {len(train_loader)} | Val batches: {len(val_loader)}")

# ── Sanity check ────────────────────────────────────────────────────
imgs, lbls = next(iter(train_loader))

print("\nSanity check:")
print("Images:", imgs.shape)
print("Labels:", lbls.shape)
print("Pixel range:", imgs.min().item(), imgs.max().item())
print("Avg positives:", lbls.sum(1).mean().item())

print("\n✅ DataLoader working correctly")

Path check:
✓ Images: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/images/images_normalized
✓ Proj CSV: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_projections.csv
✓ Report CSV: /kaggle/input/datasets/raddar/chest-xrays-indiana-university/indiana_reports.csv

Valid records: 3818

Label distribution:
atelectasis         : 362
consolidation       : 1201
infiltration        : 411
pneumothorax        : 2454
edema               : 333
emphysema           : 127
fibrosis            : 27
effusion            : 2825
pneumonia           : 247
pleural_thickening  : 37
cardiomegaly        : 280
nodule              : 282
mass                : 186
hernia              : 53

Train: 3054 | Val: 764

Train batches: 96 | Val batches: 24

Sanity check:
Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32, 14])
Pixel range: -2.1179039478302 2.640000104904175
Avg positives: 2.71875

✅ DataLoader working correctly


## Cell 6 — Load Model (Stage 1-4 Original, Head → 14 Classes)

In [15]:
print("Loading MambaVision-T (unmodified baseline)...")
try:
    model = mamba_vision_T(pretrained=True, num_classes=1000)
    print("✓ Pretrained ImageNet weights loaded")
except Exception as e:
    print(f"⚠ Pretrained failed ({e}) — using random init")
    model = mamba_vision_T(pretrained=False, num_classes=1000)

in_features = model.head.in_features
model.head  = nn.Linear(in_features, 14, bias=True)
nn.init.trunc_normal_(model.head.weight, std=0.02)
nn.init.zeros_(model.head.bias)

for p in model.parameters(): p.requires_grad = True
model = model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

EXP_NAME = f'baseline_indiana_{datetime.now().strftime("%Y%m%d_%H%M%S")}'

print(f"Stage 1 : {type(model.levels[0]).__name__}  (ConvBlock)")
print(f"Stage 2 : {type(model.levels[1]).__name__}  (ConvBlock)")
print(f"Stage 3 : {type(model.levels[2]).__name__}  (Mamba+Attn)")
print(f"Stage 4 : {type(model.levels[3]).__name__}  (Mamba+Attn — ORIGINAL)")
print(f"\nTotal params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Model size FP32  : {total_params*4/1024**2:.2f} MB")
print(f"\n✓ Model on {device}  |  Experiment: {EXP_NAME}")

Loading MambaVision-T (unmodified baseline)...
✓ Pretrained ImageNet weights loaded
Stage 1 : MambaVisionLayer  (ConvBlock)
Stage 2 : MambaVisionLayer  (ConvBlock)
Stage 3 : MambaVisionLayer  (Mamba+Attn)
Stage 4 : MambaVisionLayer  (Mamba+Attn — ORIGINAL)

Total params     : 31,162,222
Trainable params : 31,162,222
Model size FP32  : 118.87 MB

✓ Model on cuda  |  Experiment: baseline_indiana_20260503_104159


## Cell 7 — FLOPs & Parameter Count
> Architecture metric — before training.

In [16]:
model.eval()
dummy_input = torch.randn(1, 3, 224, 224).to(device)
GFLOPS = 0.0

if FLOPS_AVAILABLE:
    with FlopCounterMode(model, display=False) as fc:
        with torch.no_grad():
            model(dummy_input)
    flops  = fc.get_total_flops()
    GFLOPS = flops / 1e9
    flops_method = "fvcore (exact)"
elif THOP_AVAILABLE:
    flops, _ = thop_profile(model, inputs=(dummy_input,), verbose=False)
    GFLOPS   = flops / 1e9
    flops_method = "thop (approx)"
else:
    GFLOPS = 4.5
    flops_method = "paper estimate"

print(f"FLOPs method     : {flops_method}")
print(f"GFLOPs / image   : {GFLOPS:.4f}")
print(f"Total params     : {total_params:,}")
print(f"Trainable params : {trainable_params:,}")
print(f"Frozen params    : {frozen_params:,}")
print(f"Model size FP32  : {total_params*4/1024**2:.2f} MB")

FLOPs method     : paper estimate
GFLOPs / image   : 4.5000
Total params     : 31,162,222
Trainable params : 31,162,222
Frozen params    : 0
Model size FP32  : 118.87 MB


## Cell 8 — Loss / Optimiser / Scheduler
> `pos_weight` computed dynamically from actual Indiana label distribution.

In [17]:
NUM_EPOCHS    = 50
LR_BACKBONE   = 1e-4
LR_HEAD       = 5e-4
WEIGHT_DECAY  = 1e-4
GRAD_CLIP     = 1.0
WARMUP_EPOCHS = 5
USE_AMP       = True
PATIENCE      = 7
MIN_DELTA     = 1e-4

# pos_weight = N_neg/N_pos per class
# Recompute from actual Indiana dataset label distribution
all_labels = np.array([r['labels'] for r in train_recs])
n_pos = all_labels.sum(0).clip(min=1)
n_neg = len(all_labels) - n_pos
pw    = (n_neg / n_pos).clip(max=300).astype(np.float32)
pos_weight = torch.tensor(pw, device=device)
print(f"pos_weight range: [{pw.min():.1f}, {pw.max():.1f}]")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

param_groups = [
    {'params': [p for n,p in model.named_parameters() if 'head' not in n],
     'lr': LR_BACKBONE, 'weight_decay': WEIGHT_DECAY},
    {'params': list(model.head.parameters()),
     'lr': LR_HEAD, 'weight_decay': 0.0},
]
optimizer = optim.AdamW(param_groups, betas=(0.9,0.999), eps=1e-8)

warmup_sched = torch.optim.lr_scheduler.LinearLR(
    optimizer, start_factor=0.01, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine_sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=NUM_EPOCHS-WARMUP_EPOCHS, eta_min=1e-7)
scheduler = torch.optim.lr_scheduler.SequentialLR(
    optimizer, schedulers=[warmup_sched, cosine_sched], milestones=[WARMUP_EPOCHS])

scaler = GradScaler('cuda', enabled=USE_AMP)

CLASS_NAMES = [
    'Atelectasis','Consolidation','Infiltration','Pneumothorax',
    'Edema','Emphysema','Fibrosis','Effusion','Pneumonia',
    'Pleural_Thickening','Cardiomegaly','Nodule','Mass','Hernia'
]
NUM_CLASSES = len(CLASS_NAMES)

SAVE_DIR = os.path.join(CHECKPOINT_DIR, EXP_NAME)
os.makedirs(SAVE_DIR, exist_ok=True)

print(f"\nExperiment      : {EXP_NAME}")
print(f"Checkpoint dir  : {SAVE_DIR}")
print(f"Backbone LR     : {LR_BACKBONE:.2e}")
print(f"Head LR         : {LR_HEAD:.2e}")
print(f"Warmup          : {WARMUP_EPOCHS} ep → Cosine to ep {NUM_EPOCHS}")
print(f"Mixed precision : {USE_AMP}")
print("✓ Loss / Optimiser / Scheduler ready")

pos_weight range: [0.4, 151.7]

Experiment      : baseline_indiana_20260503_104159
Checkpoint dir  : /kaggle/working/checkpoints/baseline_indiana_20260503_104159
Backbone LR     : 1.00e-04
Head LR         : 5.00e-04
Warmup          : 5 ep → Cosine to ep 30
Mixed precision : True
✓ Loss / Optimiser / Scheduler ready


## Cell 9 — Training & Validation Functions

In [18]:
class AverageMeter:
    def __init__(self): self.reset()
    def reset(self): self.val=self.avg=self.sum=self.count=0
    def update(self,val,n=1):
        self.val=val; self.sum+=val*n; self.count+=n; self.avg=self.sum/self.count


def train_epoch(model, loader, criterion, optimizer, scaler, device, epoch, accum=4):
    model.train()
    meter = AverageMeter()
    optimizer.zero_grad()
    for i, (imgs, tgts) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        tgts = tgts.to(device, non_blocking=True)
        with autocast('cuda', enabled=USE_AMP):
            out  = model(imgs)
            loss = criterion(out, tgts) / accum
        scaler.scale(loss).backward()
        if (i+1) % accum == 0:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
        meter.update(loss.item()*accum, imgs.size(0))
        if (i+1) % max(1, len(loader)//3) == 0 or (i+1) == len(loader):
            print(f"  Ep[{epoch}][{i+1}/{len(loader)}] loss={meter.avg:.4f}"
                  f"  lr={optimizer.param_groups[0]['lr']:.2e}", flush=True)
    return meter.avg


def validate(model, loader, criterion, device):
    model.eval()
    meter = AverageMeter()
    probs_all, tgts_all = [], []
    with torch.no_grad():
        for imgs, tgts in loader:
            imgs = imgs.to(device, non_blocking=True)
            tgts = tgts.to(device, non_blocking=True)
            with autocast('cuda', enabled=USE_AMP):
                out  = model(imgs)
                loss = criterion(out, tgts)
            meter.update(loss.item(), imgs.size(0))
            probs_all.append(torch.sigmoid(out).cpu().numpy())
            tgts_all.append(tgts.cpu().numpy())

    probs = np.concatenate(probs_all)
    tgts  = np.concatenate(tgts_all)
    preds = (probs >= 0.5).astype(int)

    acc     = (preds == tgts).mean()
    emr     = (preds == tgts).all(axis=1).mean()
    hl      = hamming_loss(tgts, preds)
    prec_mi = precision_score(tgts, preds, average='micro',    zero_division=0)
    rec_mi  = recall_score(   tgts, preds, average='micro',    zero_division=0)
    f1_mi   = f1_score(       tgts, preds, average='micro',    zero_division=0)
    f1_ma   = f1_score(       tgts, preds, average='macro',    zero_division=0)
    f1_wt   = f1_score(       tgts, preds, average='weighted', zero_division=0)
    f1_pc   = f1_score(       tgts, preds, average=None,       zero_division=0)
    prec_pc = precision_score(tgts, preds, average=None,       zero_division=0)
    rec_pc  = recall_score(   tgts, preds, average=None,       zero_division=0)

    spec_pc = np.zeros(tgts.shape[1])
    for ci in range(tgts.shape[1]):
        tn = ((preds[:,ci]==0)&(tgts[:,ci]==0)).sum()
        fp = ((preds[:,ci]==1)&(tgts[:,ci]==0)).sum()
        spec_pc[ci] = tn/(tn+fp) if (tn+fp)>0 else 0.0

    mcc_pc   = np.array([matthews_corrcoef(tgts[:,ci], preds[:,ci])
                          for ci in range(tgts.shape[1])])
    mcc_mean = float(np.mean(mcc_pc))

    try:
        auc_ma = roc_auc_score(tgts, probs, average='macro')
        auc_mi = roc_auc_score(tgts, probs, average='micro')
        auc_pc = roc_auc_score(tgts, probs, average=None)
    except Exception:
        auc_ma = auc_mi = 0.0; auc_pc = np.zeros(NUM_CLASSES)

    try:
        map_score = average_precision_score(tgts, probs, average='macro')
        ap_pc     = average_precision_score(tgts, probs, average=None)
    except Exception:
        map_score = 0.0; ap_pc = np.zeros(NUM_CLASSES)

    return {
        'loss': meter.avg, 'accuracy': float(acc),
        'emr': float(emr),  'hamming': float(hl),
        'precision': float(prec_mi), 'recall': float(rec_mi),
        'f1_micro': float(f1_mi),    'f1_macro': float(f1_ma),
        'f1_weighted': float(f1_wt), 'auc_macro': float(auc_ma),
        'auc_micro': float(auc_mi),  'map': float(map_score),
        'mcc': float(mcc_mean),
        'f1_per': f1_pc,   'prec_per': prec_pc, 'rec_per':  rec_pc,
        'spec_per': spec_pc,'auc_per': auc_pc,   'ap_per':   ap_pc,
        'mcc_per': mcc_pc,  'probs': probs,       'targets':  tgts,
    }

print("✓ train_epoch + validate defined")

✓ train_epoch + validate defined


## Cell 10 — Training Loop
> GPU peak training memory tracked every epoch.

In [19]:
from IPython.display import clear_output

history = {k: [] for k in [
    'epoch','train_loss','val_loss',
    'accuracy','emr','hamming',
    'precision','recall',
    'f1_micro','f1_macro','f1_weighted',
    'auc_macro','auc_micro','map','mcc',
    'lr_backbone','lr_head'
]}

best_f1 = 0.0; patience_cnt = 0; PEAK_TRAIN_MB = 0.0

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()

LOG_FILE = os.path.join(SAVE_DIR, 'training_log.txt')
log_f = open(LOG_FILE, 'w')
print(f"Training [BASELINE] — {NUM_EPOCHS} epochs")
log_f.write(f"Training [BASELINE] — {NUM_EPOCHS} epochs\n")

for epoch in range(1, NUM_EPOCHS+1):
    t_loss = train_epoch(model, train_loader, criterion, optimizer,
                         scaler, device, epoch, accum=ACCUM_STEPS)
    if torch.cuda.is_available():
        peak_now = torch.cuda.max_memory_allocated()/1024**2
        if peak_now > PEAK_TRAIN_MB: PEAK_TRAIN_MB = peak_now
    v = validate(model, val_loader, criterion, device)
    scheduler.step()
    lr_bb = optimizer.param_groups[0]['lr']
    lr_hd = optimizer.param_groups[1]['lr']

    clear_output(wait=True)
    print(f"Training [BASELINE] — {NUM_EPOCHS} epochs")
    print(f"{'='*70}")
    print(f"  Epoch {epoch}/{NUM_EPOCHS}  lr_bb={lr_bb:.2e}  patience={patience_cnt}/{PATIENCE}")
    print(f"{'='*70}")
    print(f"  Train loss   : {t_loss:.4f}")
    print(f"  Val   loss   : {v['loss']:.4f}")
    print(f"  Accuracy     : {v['accuracy']:.4f}")
    print(f"  EMR          : {v['emr']:.4f}")
    print(f"  Hamming      : {v['hamming']:.4f}  ↓")
    print(f"  Precision    : {v['precision']:.4f}")
    print(f"  Recall       : {v['recall']:.4f}")
    print(f"  F1 micro     : {v['f1_micro']:.4f}")
    print(f"  F1 macro     : {v['f1_macro']:.4f}")
    print(f"  F1 weighted  : {v['f1_weighted']:.4f}")
    print(f"  AUC-ROC mac  : {v['auc_macro']:.4f}")
    print(f"  AUC-ROC mic  : {v['auc_micro']:.4f}")
    print(f"  mAP (AUC-PR) : {v['map']:.4f}")
    print(f"  MCC          : {v['mcc']:.4f}")
    if torch.cuda.is_available():
        print(f"  Peak VRAM    : {PEAK_TRAIN_MB:.1f} MB")

    # Running epoch table
    print(f"\n{'─'*75}")
    print(f"  {'Ep':>3} {'TrLoss':>7} {'VaLoss':>7} {'Acc':>6} {'F1mic':>6} {'F1mac':>6} {'AUC':>6} {'mAP':>6} {'MCC':>6}")
    print(f"{'─'*75}")
    for ep_i in range(len(history['epoch'])):
        print(f"  {history['epoch'][ep_i]:>3} {history['train_loss'][ep_i]:>7.4f} "
              f"{history['val_loss'][ep_i]:>7.4f} {history['accuracy'][ep_i]:>6.4f} "
              f"{history['f1_micro'][ep_i]:>6.4f} {history['f1_macro'][ep_i]:>6.4f} "
              f"{history['auc_macro'][ep_i]:>6.4f} {history['map'][ep_i]:>6.4f} "
              f"{history['mcc'][ep_i]:>6.4f}")
    # Print current epoch last
    print(f"  {epoch:>3} {t_loss:>7.4f} {v['loss']:>7.4f} {v['accuracy']:>6.4f} "
          f"{v['f1_micro']:>6.4f} {v['f1_macro']:>6.4f} "
          f"{v['auc_macro']:>6.4f} {v['map']:>6.4f} {v['mcc']:>6.4f}  ← current")

    for k,val_ in [
        ('epoch',epoch),('train_loss',t_loss),('val_loss',v['loss']),
        ('accuracy',v['accuracy']),('emr',v['emr']),('hamming',v['hamming']),
        ('precision',v['precision']),('recall',v['recall']),
        ('f1_micro',v['f1_micro']),('f1_macro',v['f1_macro']),
        ('f1_weighted',v['f1_weighted']),('auc_macro',v['auc_macro']),
        ('auc_micro',v['auc_micro']),('map',v['map']),('mcc',v['mcc']),
        ('lr_backbone',lr_bb),('lr_head',lr_hd),
    ]:
        history[k].append(val_)

    log_line = (f"Ep{epoch:03d} | loss={t_loss:.4f} | val={v['loss']:.4f} | "
                f"acc={v['accuracy']:.4f} | f1={v['f1_micro']:.4f} | "
                f"auc={v['auc_macro']:.4f} | map={v['map']:.4f} | mcc={v['mcc']:.4f}")
    log_f.write(log_line + "\n"); log_f.flush()

    if v['f1_micro'] > best_f1 + MIN_DELTA:
        best_f1 = v['f1_micro']; patience_cnt = 0
        best_path = os.path.join(SAVE_DIR, 'best_model.pth')
        torch.save({
            'epoch': epoch, 'best_f1': best_f1,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'val_metrics': {k: v[k] for k in [
                'loss','accuracy','emr','hamming','precision','recall',
                'f1_micro','f1_macro','f1_weighted',
                'auc_macro','auc_micro','map','mcc']},
            'history': history,
        }, best_path)
        print(f"  ★ Best model saved  F1={best_f1:.4f}")
    else:
        patience_cnt += 1

    if epoch % 5 == 0:
        ckpt = os.path.join(SAVE_DIR, f'checkpoint_ep{epoch:03d}.pth')
        torch.save({'epoch':epoch,'model_state_dict':model.state_dict(),
                     'history':history}, ckpt)

    if patience_cnt >= PATIENCE:
        print(f"  Early stopping at epoch {epoch}")
        break

log_f.close()
print(f"\nTraining done | Best F1 (micro): {best_f1:.4f}")
print(f"Peak training VRAM: {PEAK_TRAIN_MB:.2f} MB")

Training [BASELINE] — 30 epochs
  Epoch 30/30  lr_bb=1.00e-07  patience=4/7
  Train loss   : 0.6600
  Val   loss   : 1.0049
  Accuracy     : 0.7380
  EMR          : 0.0406
  Hamming      : 0.2620  ↓
  Precision    : 0.3342
  Recall       : 0.5849
  F1 micro     : 0.4253
  F1 macro     : 0.3026
  F1 weighted  : 0.5254
  AUC-ROC mac  : 0.6987
  AUC-ROC mic  : 0.7661
  mAP (AUC-PR) : 0.3231
  MCC          : 0.1816
  Peak VRAM    : 1641.5 MB

───────────────────────────────────────────────────────────────────────────
   Ep  TrLoss  VaLoss    Acc  F1mic  F1mac    AUC    mAP    MCC
───────────────────────────────────────────────────────────────────────────
    1  1.1600  1.1650 0.5296 0.2650 0.1930 0.4995 0.1815 0.0239
    2  1.1518  1.1479 0.5423 0.2888 0.2134 0.5833 0.1989 0.0594
    3  1.1330  1.1192 0.5914 0.2871 0.2226 0.6197 0.2291 0.0814
    4  1.1054  1.0842 0.5212 0.2873 0.2233 0.6695 0.2432 0.0927
    5  1.0543  1.0167 0.6163 0.3463 0.2493 0.6971 0.2639 0.1213
    6  1.0417  1.1077

## Cell 11 — Load Best Model

In [20]:
best_path = os.path.join(SAVE_DIR, 'best_model.pth')
if os.path.exists(best_path):
    ckpt = torch.load(best_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"✓ Best model loaded — Epoch {ckpt['epoch']}  F1={ckpt['best_f1']:.4f}")
else:
    print("⚠ No best_model.pth — using current weights")
model.eval()
print("  eval() mode set")

✓ Best model loaded — Epoch 30  F1=0.4253
  eval() mode set


## Cell 12 — Inference Time
> After training, warm GPU, 100 benchmark runs.

In [21]:
WARMUP_RUNS = 10; BENCH_RUNS = 100
model.eval()
single_inp = torch.randn(1, 3, 224, 224).to(device)
batch_inp  = torch.randn(BATCH_SIZE, 3, 224, 224).to(device)

with torch.no_grad():
    for _ in range(WARMUP_RUNS): _ = model(single_inp)
if torch.cuda.is_available(): torch.cuda.synchronize()

times_single = []
with torch.no_grad():
    for _ in range(BENCH_RUNS):
        t0 = time.perf_counter()
        _ = model(single_inp)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times_single.append((time.perf_counter()-t0)*1000)

T_MEAN = float(np.mean(times_single)); T_STD  = float(np.std(times_single))
T_MIN  = float(np.min(times_single));  T_MAX  = float(np.max(times_single))
T_P95  = float(np.percentile(times_single, 95))
FPS_SINGLE = 1000.0 / T_MEAN

print(f"Inference — single image ({BENCH_RUNS} runs, warm GPU):")
print(f"  Mean : {T_MEAN:.3f} ms  |  Std : {T_STD:.3f} ms")
print(f"  Min  : {T_MIN:.3f} ms  |  P95 : {T_P95:.3f} ms")
print(f"  FPS  : {FPS_SINGLE:.1f} imgs/sec")

with torch.no_grad():
    for _ in range(WARMUP_RUNS): _ = model(batch_inp)
if torch.cuda.is_available(): torch.cuda.synchronize()

times_batch = []
with torch.no_grad():
    for _ in range(BENCH_RUNS):
        t0 = time.perf_counter()
        _ = model(batch_inp)
        if torch.cuda.is_available(): torch.cuda.synchronize()
        times_batch.append((time.perf_counter()-t0)*1000)

BT_MEAN    = float(np.mean(times_batch))
BT_PER_IMG = BT_MEAN / BATCH_SIZE
FPS_BATCH  = BATCH_SIZE * 1000.0 / BT_MEAN
print(f"\nInference — batch={BATCH_SIZE}: {BT_MEAN:.3f} ms total  |  {BT_PER_IMG:.3f} ms/img  |  {FPS_BATCH:.1f} FPS")
print("✓ Inference benchmark complete")

Inference — single image (100 runs, warm GPU):
  Mean : 11.296 ms  |  Std : 0.273 ms
  Min  : 10.947 ms  |  P95 : 11.750 ms
  FPS  : 88.5 imgs/sec

Inference — batch=32: 121.435 ms total  |  3.795 ms/img  |  263.5 FPS
✓ Inference benchmark complete


## Cell 13 — GPU Memory
> Inference peak (after training) + training peak (from Cell 10).

In [22]:
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(); torch.cuda.empty_cache()
    model.eval()
    with torch.no_grad():
        _ = model(torch.randn(1,3,224,224).to(device))
    torch.cuda.synchronize()
    INFER_ALLOC_MB = torch.cuda.memory_allocated()   / 1024**2
    INFER_CACHE_MB = torch.cuda.memory_reserved()    / 1024**2
    INFER_PEAK_MB  = torch.cuda.max_memory_allocated()/ 1024**2
    print(f"GPU Memory — INFERENCE (1 image):")
    print(f"  Allocated : {INFER_ALLOC_MB:.2f} MB  |  Cached : {INFER_CACHE_MB:.2f} MB")
    print(f"  Peak      : {INFER_PEAK_MB:.2f} MB   |  Util   : {INFER_PEAK_MB/TOTAL_VRAM_MB*100:.1f}%")
    print(f"\nGPU Memory — TRAINING PEAK:")
    print(f"  Peak      : {PEAK_TRAIN_MB:.2f} MB   |  Util   : {PEAK_TRAIN_MB/TOTAL_VRAM_MB*100:.1f}%")
    print(f"  Total VRAM: {TOTAL_VRAM_MB:.2f} MB")
else:
    INFER_ALLOC_MB = INFER_CACHE_MB = INFER_PEAK_MB = 0.0
    print("⚠ CUDA not available")
print("✓ GPU memory profiled")

GPU Memory — INFERENCE (1 image):
  Allocated : 764.34 MB  |  Cached : 798.00 MB
  Peak      : 769.27 MB   |  Util   : 5.2%

GPU Memory — TRAINING PEAK:
  Peak      : 1641.52 MB   |  Util   : 11.0%
  Total VRAM: 14912.69 MB
✓ GPU memory profiled


## Cell 14 — Final Evaluation — All 13 Metrics

In [23]:
print("Running final evaluation...")
final = validate(model, val_loader, criterion, device)

print(f"\n{'─'*62}")
print(f"  {'METRIC':<30} {'VALUE':>10}")
print(f"  {'─'*42}")
for lbl, key in [
    ('Loss',                    'loss'),
    ('Accuracy (element-wise)', 'accuracy'),
    ('Exact Match Ratio (EMR)', 'emr'),
    ('Hamming Loss ↓',          'hamming'),
    ('Precision (micro)',       'precision'),
    ('Recall (micro)',          'recall'),
    ('F1 (micro)',              'f1_micro'),
    ('F1 (macro)',              'f1_macro'),
    ('F1 (weighted)',           'f1_weighted'),
    ('AUC-ROC (macro)',         'auc_macro'),
    ('AUC-ROC (micro)',         'auc_micro'),
    ('mAP / AUC-PR (macro)',    'map'),
    ('MCC (mean)',              'mcc'),
]:
    print(f"  {lbl:<30} {final[key]:>10.4f}")
print(f"  {'─'*42}")

print(f"\nPer-class (F1 / AUC / AP / Prec / Rec / Spec / MCC):")
print(f"  {'Class':<22} {'F1':>6} {'AUC':>6} {'AP':>6} {'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6}")
print(f"  {'─'*64}")
for i, cn in enumerate(CLASS_NAMES):
    print(f"  {cn:<22} {final['f1_per'][i]:>6.3f} {final['auc_per'][i]:>6.3f}"
          f" {final['ap_per'][i]:>6.3f} {final['prec_per'][i]:>6.3f}"
          f" {final['rec_per'][i]:>6.3f} {final['spec_per'][i]:>6.3f}"
          f" {final['mcc_per'][i]:>6.3f}")

Running final evaluation...

──────────────────────────────────────────────────────────────
  METRIC                              VALUE
  ──────────────────────────────────────────
  Loss                               1.0049
  Accuracy (element-wise)            0.7380
  Exact Match Ratio (EMR)            0.0406
  Hamming Loss ↓                     0.2620
  Precision (micro)                  0.3342
  Recall (micro)                     0.5849
  F1 (micro)                         0.4253
  F1 (macro)                         0.3026
  F1 (weighted)                      0.5254
  AUC-ROC (macro)                    0.6987
  AUC-ROC (micro)                    0.7661
  mAP / AUC-PR (macro)               0.3231
  MCC (mean)                         0.1816
  ──────────────────────────────────────────

Per-class (F1 / AUC / AP / Prec / Rec / Spec / MCC):
  Class                      F1    AUC     AP   Prec    Rec   Spec    MCC
  ────────────────────────────────────────────────────────────────
  Atele

## Cell 15 — Training Curves (3×3)

In [24]:
ep = history['epoch']
fig, axes = plt.subplots(3, 3, figsize=(20, 15))

def plot(ax, keys, title, logy=False):
    styles=[('o-','steelblue'),('s-','coral'),('^-','green'),('d-','purple')]
    for (key,lbl),(mk,col) in zip(keys,styles):
        ax.plot(ep, history[key], mk, ms=3, color=col, label=lbl)
    ax.set_title(title, fontweight='bold'); ax.set_xlabel('Epoch')
    ax.legend(fontsize=8); ax.grid(alpha=.3)
    if logy: ax.set_yscale('log')

plot(axes[0,0],[('train_loss','Train'),('val_loss','Val')],'Loss')
plot(axes[0,1],[('f1_micro','micro'),('f1_macro','macro'),('f1_weighted','weighted')],'F1 Score')
plot(axes[0,2],[('auc_macro','AUC-ROC mac'),('auc_micro','AUC-ROC mic'),('map','mAP')],'AUC & mAP')
plot(axes[1,0],[('precision','Precision'),('recall','Recall')],'Precision & Recall')
plot(axes[1,1],[('accuracy','Accuracy'),('emr','Exact Match')],'Accuracy & EMR')
plot(axes[1,2],[('mcc','MCC'),('hamming','Hamming↓')],'MCC & Hamming Loss')
plot(axes[2,0],[('lr_backbone','Backbone'),('lr_head','Head')],'Learning Rate',logy=True)

axes[2,1].scatter(history['recall'],history['precision'],
                  c=history['epoch'],cmap='viridis',s=30,zorder=3)
axes[2,1].set_xlabel('Recall'); axes[2,1].set_ylabel('Precision')
axes[2,1].set_title('Precision vs Recall (epochs)',fontweight='bold'); axes[2,1].grid(alpha=.3)

axes[2,2].scatter(history['f1_micro'],history['auc_macro'],
                  c=history['epoch'],cmap='plasma',s=30,zorder=3)
axes[2,2].set_xlabel('F1 micro'); axes[2,2].set_ylabel('AUC-ROC')
axes[2,2].set_title('F1 vs AUC-ROC (epochs)',fontweight='bold'); axes[2,2].grid(alpha=.3)

plt.suptitle('MambaVision-T Baseline | Indiana Chest X-ray (Kaggle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show(); print("✓ Training curves saved")

✓ Training curves saved


## Cell 16 — Per-Class Visualisation

In [25]:
fig, axes = plt.subplots(3, 3, figsize=(24, 20))

def hbar(ax, vals, title, color):
    idx = np.argsort(vals)
    vs  = vals[idx]; ns = [CLASS_NAMES[i] for i in idx]
    bars = ax.barh(ns, vs, color=color, edgecolor='white', alpha=0.85)
    ax.set_xlim([0,1]); ax.set_title(title, fontweight='bold', fontsize=11)
    mv = np.mean(vals)
    ax.axvline(mv, color='black', ls='--', lw=1.2, label=f'Mean={mv:.3f}')
    ax.legend(fontsize=8)
    for bar, v in zip(bars, vs):
        ax.text(min(v+.01, 0.93), bar.get_y()+bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=8)
    ax.grid(axis='x', alpha=.3)

hbar(axes[0,0],final['f1_per'],  'Per-Class F1',         'steelblue')
hbar(axes[0,1],final['auc_per'], 'Per-Class AUC-ROC',    'coral')
hbar(axes[0,2],final['ap_per'],  'Per-Class AP (AUC-PR)','darkorange')
hbar(axes[1,0],final['prec_per'],'Per-Class Precision',  'mediumseagreen')
hbar(axes[1,1],final['rec_per'], 'Per-Class Recall',     'mediumpurple')
hbar(axes[1,2],final['spec_per'],'Per-Class Specificity','teal')
hbar(axes[2,0],final['mcc_per'], 'Per-Class MCC',        'saddlebrown')

metric_names = ['F1','AUC-ROC','AP','Precision','Recall','Specificity','MCC']
metric_vals  = [np.mean(final['f1_per']),np.mean(final['auc_per']),
                np.mean(final['ap_per']),np.mean(final['prec_per']),
                np.mean(final['rec_per']),np.mean(final['spec_per']),
                np.clip(np.mean(final['mcc_per']),0,1)]
angles = np.linspace(0,2*np.pi,len(metric_names),endpoint=False).tolist()
vr = metric_vals+[metric_vals[0]]; ar = angles+[angles[0]]
axes[2,1].remove()
ax_r = fig.add_subplot(3,3,8,polar=True)
ax_r.plot(ar,vr,'o-',lw=2,color='steelblue'); ax_r.fill(ar,vr,alpha=0.25,color='steelblue')
ax_r.set_xticks(angles); ax_r.set_xticklabels(metric_names,fontsize=9)
ax_r.set_ylim(0,1); ax_r.set_title('Mean Metrics Radar',fontweight='bold',pad=15)

hdata = np.array([final['f1_per'],final['auc_per'],final['ap_per'],
                  final['prec_per'],final['rec_per'],final['spec_per'],
                  np.clip(final['mcc_per'],0,1)])
ax_h = axes[2,2]
im = ax_h.imshow(hdata, aspect='auto', cmap='RdYlGn', vmin=0, vmax=1)
ax_h.set_xticks(range(len(CLASS_NAMES)))
ax_h.set_xticklabels(CLASS_NAMES, rotation=45, ha='right', fontsize=7)
ax_h.set_yticks(range(7)); ax_h.set_yticklabels(metric_names, fontsize=9)
ax_h.set_title('Per-Class Heatmap', fontweight='bold')
plt.colorbar(im, ax=ax_h, fraction=0.03)
for r in range(7):
    for col in range(len(CLASS_NAMES)):
        ax_h.text(col,r,f'{hdata[r,col]:.2f}',ha='center',va='center',fontsize=6)

plt.suptitle('MambaVision-T Baseline | Indiana Chest X-ray (Kaggle)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR,'per_class_metrics.png'),dpi=150,bbox_inches='tight')
plt.show(); print("✓ Per-class visualisation saved")

✓ Per-class visualisation saved


## Cell 17 — Full Benchmark Summary Table

In [26]:
SEP = "─"*65
print("\n"+"="*65)
print(f"  MAMBAVISION-T  |  Indiana Chest X-ray  |  BASELINE")
print(f"  Dataset  : Indiana University Chest X-ray (Kaggle)")
print(f"  Stage 4  : MambaVisionLayer (original — UNMODIFIED)")
print("="*65)
print(f"\n{SEP}")
print("  MODEL COMPLEXITY")
print(SEP)
print(f"  Total parameters         : {total_params:>14,}")
print(f"  Trainable parameters     : {trainable_params:>14,}")
print(f"  Frozen parameters        : {frozen_params:>14,}")
print(f"  Model size  (FP32)       : {total_params*4/1024**2:>10.2f} MB")
print(f"  GFLOPs per image         : {GFLOPS:>10.4f}")
print(f"  FLOPs method             : {flops_method}")
print(f"\n{SEP}")
print("  INFERENCE TIME  (warm GPU, 100 runs)")
print(SEP)
print(f"  Single image — mean      : {T_MEAN:>8.3f} ms")
print(f"  Single image — std       : {T_STD:>8.3f} ms")
print(f"  Single image — P95       : {T_P95:>8.3f} ms")
print(f"  Single image — FPS       : {FPS_SINGLE:>8.1f} imgs/sec")
print(f"  Batch (bs={BATCH_SIZE}) per img    : {BT_PER_IMG:>8.3f} ms")
print(f"  Batch (bs={BATCH_SIZE}) FPS        : {FPS_BATCH:>8.1f} imgs/sec")
if torch.cuda.is_available():
    print(f"\n{SEP}")
    print("  GPU MEMORY")
    print(SEP)
    print(f"  Training peak VRAM       : {PEAK_TRAIN_MB:>8.2f} MB  (fwd+bwd+optim)")
    print(f"  Inference peak VRAM      : {INFER_PEAK_MB:>8.2f} MB  (fwd only)")
    print(f"  Training utilisation     : {PEAK_TRAIN_MB/TOTAL_VRAM_MB*100:>7.1f}%")
    print(f"  Inference utilisation    : {INFER_PEAK_MB/TOTAL_VRAM_MB*100:>7.1f}%")
    print(f"  Total VRAM               : {TOTAL_VRAM_MB:>8.2f} MB")
print(f"\n{SEP}")
print("  CLASSIFICATION METRICS  (val set, best model)")
print(SEP)
for lbl, key in [
    ('Accuracy (element-wise)', 'accuracy'),
    ('Exact Match Ratio',       'emr'),
    ('Hamming Loss ↓',          'hamming'),
    ('Precision  (micro)',      'precision'),
    ('Recall     (micro)',      'recall'),
    ('F1         (micro)',      'f1_micro'),
    ('F1         (macro)',      'f1_macro'),
    ('F1         (weighted)',   'f1_weighted'),
    ('AUC-ROC    (macro)',      'auc_macro'),
    ('AUC-ROC    (micro)',      'auc_micro'),
    ('mAP / AUC-PR (macro)',    'map'),
    ('MCC        (mean)',       'mcc'),
]:
    print(f"  {lbl:<30} : {final[key]:>8.4f}")
print(f"\n{SEP}")
print("  PER-CLASS METRICS")
print(SEP)
print(f"  {'Class':<22} {'F1':>6} {'AUC':>6} {'AP':>6} {'Prec':>6} {'Rec':>6} {'Spec':>6} {'MCC':>6}")
print(f"  {'─'*64}")
for i, cn in enumerate(CLASS_NAMES):
    print(f"  {cn:<22} {final['f1_per'][i]:>6.3f} {final['auc_per'][i]:>6.3f}"
          f" {final['ap_per'][i]:>6.3f} {final['prec_per'][i]:>6.3f}"
          f" {final['rec_per'][i]:>6.3f} {final['spec_per'][i]:>6.3f}"
          f" {final['mcc_per'][i]:>6.3f}")
print("="*65)


  MAMBAVISION-T  |  Indiana Chest X-ray  |  BASELINE
  Dataset  : Indiana University Chest X-ray (Kaggle)
  Stage 4  : MambaVisionLayer (original — UNMODIFIED)

─────────────────────────────────────────────────────────────────
  MODEL COMPLEXITY
─────────────────────────────────────────────────────────────────
  Total parameters         :     31,162,222
  Trainable parameters     :     31,162,222
  Frozen parameters        :              0
  Model size  (FP32)       :     118.87 MB
  GFLOPs per image         :     4.5000
  FLOPs method             : paper estimate

─────────────────────────────────────────────────────────────────
  INFERENCE TIME  (warm GPU, 100 runs)
─────────────────────────────────────────────────────────────────
  Single image — mean      :   11.296 ms
  Single image — std       :    0.273 ms
  Single image — P95       :   11.750 ms
  Single image — FPS       :     88.5 imgs/sec
  Batch (bs=32) per img    :    3.795 ms
  Batch (bs=32) FPS        :    263.5 imgs/se

## Cell 18 — Save Results (JSON + TXT → /kaggle/working/)

In [ ]:
results = {
    'experiment': EXP_NAME, 'type': 'Baseline',
    'stage4': 'MambaVisionLayer (original unmodified)', 'dataset': 'Indiana University Chest X-ray (Kaggle)',
    'generated': datetime.now().isoformat(),
    'model_complexity': {
        'total_params': int(total_params), 'trainable_params': int(trainable_params),
        'frozen_params': int(frozen_params),
        'model_size_fp32_mb': round(total_params*4/1024**2,4),
        'gflops_per_image': round(GFLOPS,4), 'flops_method': flops_method,
    },
    'inference': {
        'bench_runs': BENCH_RUNS,
        'single_ms_mean': round(T_MEAN,4), 'single_ms_std': round(T_STD,4),
        'single_ms_p95': round(T_P95,4),   'single_fps': round(FPS_SINGLE,2),
        'batch_ms_mean': round(BT_MEAN,4), 'batch_per_image_ms': round(BT_PER_IMG,4),
        'batch_fps': round(FPS_BATCH,2),   'batch_size': BATCH_SIZE,
    },
    'gpu_memory': {
        'training_peak_mb': round(PEAK_TRAIN_MB,4),
        'inference_peak_mb': round(INFER_PEAK_MB,4),
        'total_vram_mb': round(TOTAL_VRAM_MB,4),
        'train_util_pct': round(PEAK_TRAIN_MB/TOTAL_VRAM_MB*100,2) if TOTAL_VRAM_MB else 0,
        'infer_util_pct': round(INFER_PEAK_MB/TOTAL_VRAM_MB*100,2) if TOTAL_VRAM_MB else 0,
    },
    'training_config': {
        'num_epochs': NUM_EPOCHS, 'batch_size': BATCH_SIZE,
        'accum_steps': ACCUM_STEPS, 'effective_batch': BATCH_SIZE*ACCUM_STEPS,
        'lr_backbone': LR_BACKBONE, 'lr_head': LR_HEAD,
        'weight_decay': WEIGHT_DECAY, 'grad_clip': GRAD_CLIP,
        'warmup_epochs': WARMUP_EPOCHS, 'optimizer': 'AdamW',
        'scheduler': 'LinearWarmup+CosineAnnealing',
        'loss': 'BCEWithLogitsLoss+dynamic_pos_weight',
        'mixed_precision': USE_AMP, 'patience': PATIENCE,
    },
    'val_metrics': {
        'overall': {k: round(float(final[k]),4) for k in [
            'loss','accuracy','emr','hamming','precision','recall',
            'f1_micro','f1_macro','f1_weighted','auc_macro','auc_micro','map','mcc']},
        'per_class': {cn: {
            'f1':   round(float(final['f1_per'][i]),4),
            'auc':  round(float(final['auc_per'][i]),4),
            'ap':   round(float(final['ap_per'][i]),4),
            'prec': round(float(final['prec_per'][i]),4),
            'rec':  round(float(final['rec_per'][i]),4),
            'spec': round(float(final['spec_per'][i]),4),
            'mcc':  round(float(final['mcc_per'][i]),4),
        } for i,cn in enumerate(CLASS_NAMES)},
    },
    'training_history': history,
    'dataset_info': {
        'name': 'Indiana University Chest X-ray',
        'train_samples': len(train_dataset), 'val_samples': len(val_dataset),
        'input_size': [224,224], 'channels': 3, 'mean': MEAN, 'std': STD,
        'split': '80/20 stratified',
    },
}

json_path = os.path.join(SAVE_DIR, 'results.json')
with open(json_path,'w') as f: json.dump(results, f, indent=2)
print(f"✓ JSON → {json_path}")

txt_lines = [
    f"Baseline  |  Indiana University Chest X-ray  |  MambaVisionLayer (original unmodified)",
    "="*60, f"Experiment: {EXP_NAME}",
    f"Generated : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}","",
    f"GFLOPs/img    : {GFLOPS:.4f}  (method: {flops_method})",
    f"Total params  : {total_params:,}",
    f"Model size MB : {total_params*4/1024**2:.2f}","",
    f"Infer/img mean: {T_MEAN:.3f} ms  |  P95: {T_P95:.3f} ms  |  FPS: {FPS_SINGLE:.1f}",
    f"Batch FPS     : {FPS_BATCH:.1f}","",
    f"Train VRAM    : {PEAK_TRAIN_MB:.2f} MB",
    f"Infer VRAM    : {INFER_PEAK_MB:.2f} MB","",
    "CLASSIFICATION METRICS:",
] + [f"  {lbl:<30}: {final[k]:.4f}"
     for lbl,k in [('Accuracy','accuracy'),('EMR','emr'),('Hamming↓','hamming'),
                    ('Precision(micro)','precision'),('Recall(micro)','recall'),
                    ('F1 micro','f1_micro'),('F1 macro','f1_macro'),
                    ('F1 weighted','f1_weighted'),('AUC-ROC macro','auc_macro'),
                    ('mAP/AUC-PR','map'),('MCC','mcc')]]

txt_path = os.path.join(SAVE_DIR,'report.txt')
with open(txt_path,'w') as f: f.write("\n".join(txt_lines))
print(f"✓ TXT  → {txt_path}")
print(f"\nAll outputs: {SAVE_DIR}")
print("\nTo download: File → Open (Kaggle) → outputs folder")

## ✓ Baseline Complete

**Output files in `/kaggle/working/checkpoints/baseline_indiana_.../`:**
- `best_model.pth` — best checkpoint
- `results.json` — all metrics (load alongside experiment JSON to compare)
- `report.txt` — plain text summary
- `training_curves.png`
- `per_class_metrics.png`
- `training_log.txt`

**Download:** Kaggle → Output tab → Download all